<a href="https://colab.research.google.com/github/issacridhin/LabWorks/blob/LLM/2348546_LLM_Lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#importing libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as functional
from torchtext.datasets import Multi30k
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer
from torch.utils.data import DataLoader
import spacy
import math
from torchtext.data.metrics import bleu_score


/usr/local/lib/python3.10/dist-packages/torchtext/datasets/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.10/dist-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.10/dist-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext i

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 52.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Set Up Data

# Load English and German tokenizers
spacy_en = spacy.load('en_core_web_sm')
spacy_de = spacy.load('de_core_news_sm')

tokenizer_en = get_tokenizer('spacy', language='en_core_web_sm')
tokenizer_de = get_tokenizer('spacy', language='de_core_news_sm')


In [ ]:
# Define a function to yield tokens from the dataset
def yield_tokens(data_iter, tokenizer):
    for data in data_iter:
        yield tokenizer(data)


In [ ]:
# Special tokens
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2

In [ ]:
pip install portalocker


  Using cached portalocker-2.10.1-py3-none-any.whl.metadata (8.5 kB)
Using cached portalocker-2.10.1-py3-none-any.whl (18 kB)


In [ ]:
pip uninstall portalocker


Found existing installation: portalocker 2.10.1
Uninstalling portalocker-2.10.1:
  Would remove:
    /usr/local/lib/python3.10/dist-packages/portalocker-2.10.1.dist-info/*
    /usr/local/lib/python3.10/dist-packages/portalocker/*
Proceed (Y/n)? y
  Successfully uninstalled portalocker-2.10.1


In [ ]:
pip install torchdata


In [ ]:
# Load the dataset
train_data, valid_data, test_data = Multi30k(root=".data", split=('train', 'valid', 'test'), language_pair=('en', 'de'))


/usr/local/lib/python3.10/dist-packages/torchdata/datapipes/__init__.py:18: UserWarning: 
################################################################################
WARNING!
The 'datapipes', 'dataloader2' modules are deprecated and will be removed in a
future torchdata release! Please see https://github.com/pytorch/data/issues/1196
to learn more and leave feedback.
################################################################################

  deprecation_warning()


In [ ]:
# Build the vocabularies
def build_vocab(data, tokenizer):
    vocab = build_vocab_from_iterator((tokenizer(sentence) for pair in data for sentence in pair),
                                      specials=["<pad>", "<bos>", "<eos>"])
    vocab.set_default_index(PAD_IDX)
    return vocab

source_vocab = build_vocab(train_data, tokenizer_en)
target_vocab = build_vocab(train_data, tokenizer_de)

/usr/local/lib/python3.10/dist-packages/torch/utils/data/datapipes/iter/combining.py:337: UserWarning: Some child DataPipes are not exhausted when __iter__ is called. We are resetting the buffer and each child DataPipe will read from the start again.
  warnings.warn("Some child DataPipes are not exhausted when __iter__ is called. We are resetting "


In [ ]:
# Data preparation function
def tensor_transform(sentence, vocab, tokenizer):
    tokens = [BOS_IDX] + [vocab[token] for token in tokenizer(sentence)] + [EOS_IDX]
    return torch.tensor(tokens, dtype=torch.long)

def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for src, tgt in batch:
        src_batch.append(tensor_transform(src, source_vocab, tokenizer_en))
        tgt_batch.append(tensor_transform(tgt, target_vocab, tokenizer_de))
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch.to(device), tgt_batch.to(device)

In [ ]:
# Create DataLoaders
train_loader = DataLoader(train_data, batch_size=32, collate_fn=collate_fn)
valid_loader = DataLoader(valid_data, batch_size=32, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=32, collate_fn=collate_fn)

In [ ]:
# Step 2: Define the Transformer Model
class SimpleTransformer(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, emb_size=128, nhead=4, num_layers=2):
        super(SimpleTransformer, self).__init__()
        self.src_emb = nn.Embedding(source_vocab_size, emb_size)
        self.tgt_emb = nn.Embedding(target_vocab_size, emb_size)
        self.transformer = nn.Transformer(d_model=emb_size, nhead=nhead, num_encoder_layers=num_layers, num_decoder_layers=num_layers)
        self.fc_out = nn.Linear(emb_size, target_vocab_size)

    def forward(self, src, tgt):
        src = self.src_emb(src)
        tgt = self.tgt_emb(tgt)
        output = self.transformer(src, tgt)
        return self.fc_out(output)

In [ ]:
# Initialize model, optimizer, and loss function
model = SimpleTransformer(len(source_vocab), len(target_vocab)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [ ]:
# Step 3: Train the Model
def train_model(model, optimizer, criterion, train_loader):
    model.train()
    for src, tgt in train_loader:
        tgt_input = tgt[:-1, :]
        tgt_output = tgt[1:, :].reshape(-1)
        optimizer.zero_grad()
        output = model(src, tgt_input).reshape(-1, model.fc_out.out_features)
        loss = criterion(output, tgt_output)
        loss.backward()
        optimizer.step()

In [ ]:
# Train for a few epochs
for epoch in range(5):
    train_model(model, optimizer, criterion, train_loader)
    print(f"Epoch {epoch+1} completed.")

In [ ]:
# Step 4: Evaluate with BLEU Score
def evaluate_bleu(model, data_loader):
    model.eval()
    targets = []
    predictions = []
    with torch.no_grad():
        for src, tgt in data_loader:
            tgt_input = tgt[:-1, :]
            output = model(src, tgt_input)
            predicted_tokens = output.argmax(2).tolist()
            for i in range(len(predicted_tokens)):
                predictions.append(predicted_tokens[i])
                targets.append([tgt[1:, i].tolist()])
    return bleu_score(predictions, targets)


In [ ]:
# Calculate BLEU score on test data
bleu = evaluate_bleu(model, test_loader)
print(f"BLEU Score: {bleu:.2f}")

In [ ]:
# Example translation
test_sentence = "This is an example of sequence to sequence translation."
translated_sentence = translate_sentence(model, test_sentence, source_vocab, target_vocab, tokenizer_en)
print(f"Original Sentence: {test_sentence}")
print(f"Translated Sentence: {translated_sentence}")